# 03 - Stage 2, yield optimization and override learning

Three things this notebook shows:

1. assessment buys real pricing accuracy (Stage 2 beats the Stage 1 anchor),
2. the value-maximising price is *not* the highest permissible price,
3. whether human overrides created or destroyed value - done properly, which
   means correcting for the fact that overrides are not randomly assigned.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from aavas.ingestion import loader
df = loader.load_training_table()
print(f"{len(df):,} applications, {df.application_date.min():%b %Y} to {df.application_date.max():%b %Y}")


## What the assessment is worth

In [ ]:
from aavas.models.stage2.evaluate import evaluate
evaluate(df)


## The demand curve and the value curve

The conversion model carries a monotone constraint on price, so the demand curve
cannot bend the wrong way. The value curve is conversion multiplied by lifetime
economics - and it peaks *inside* the permissible band.

In [ ]:
from aavas.governance.model_registry import load_bundle
from aavas.agents.orchestrator import PricingOrchestrator

bundle = load_bundle()
orchestrator = PricingOrchestrator(bundle=bundle)
result = orchestrator.process(df.iloc[500].to_dict(), persist=False)

scenarios = pd.DataFrame(result["recommendation"]["scenario_table"])
print(f"Card rate    {result['policy']['card_rate']:.2f}%")
print(f"Recommended  {result['recommendation']['recommended_rate']:.2f}%")
print(f"Value peak   {result['recommendation']['marginal_analysis']['unconstrained_peak_rate']:.2f}%")
print()
print(scenarios[["rate", "conversion_probability", "expected_life_years",
                 "lifetime_value_pct", "expected_value_inr"]].iloc[::2].round(3).to_string(index=False))


In [ ]:
ax = scenarios.plot(x="rate", y="expected_value_inr", legend=False,
                    title="Risk-adjusted value across permissible prices", figsize=(9, 4))
ax.axvline(result["recommendation"]["recommended_rate"], color="green", linestyle="--", label="recommended")
ax.axvline(result["policy"]["card_rate"], color="grey", linestyle=":", label="card rate")
ax.set_xlabel("offered rate (%)"); ax.set_ylabel("expected value (Rs)"); ax.legend()


## Why this price and not a higher one

In [ ]:
for name, comparison in result["recommendation"]["comparisons"].items():
    print(f"  {name:<24} {comparison['rate']:.2f}%  conversion {comparison['conversion_probability']:.0%}  "
          f"value {comparison['expected_value_inr']:>10,.0f}  ({comparison['value_vs_recommended_inr']:+,.0f})")
print()
for code in result["explanation"]["reason_codes"]:
    print(f"  {code['text']:<50} {code['label']}")
print()
print(result["explanation"]["commercial_vs_risk"])


## Did human overrides create value?

The naive comparison is confounded. Salespeople discount precisely the files that
were going to be hard to close, so overridden files look worse on conversion even
when the override helped.

The correction below compares overridden and non-overridden files *within strata
of predicted override propensity*. It removes the first-order selection effect.
It is not a randomised experiment - answering this question properly needs a
price test, which is the natural next step for this POC.

In [ ]:
review = orchestrator.learning.review_overrides(df)
print("Naive:   ", review["naive_comparison"])
print("Adjusted:", {k: review["adjusted_overall"][k] for k in
                    ["s2d_lift_pts", "value_lift_inr", "value_lift_se_inr", "verdict"]})
print(f"\nThe gap between them is Rs {review['confounding_gap_inr']:,.0f} per file of pure selection effect.")
print()
pd.DataFrame(review["reason_league_table"])


## Evidenced versus unevidenced competitor claims

The policy caps the discount an *unverified* competitor claim can justify. This
is the evidence for or against that rule.

In [ ]:
pd.DataFrame(review["evidence_effect"])


In [ ]:
for suggestion in review["suggested_adjustments"]:
    print(f"{suggestion['reason']}: {suggestion['proposal']}")
    print(f"  {suggestion['evidence']}  ->  {suggestion['requires']}\n")
